# 02 — Build the working subset + quality checks

Converted from `scripts/02_subset.py` (Derrick, data prep).

Selects the agreed slice (2 cities × 5 stores × 10 products × 90 days), runs quality checks, and writes the subset (`.parquet` + daily `.csv`), the data dictionary, the subset-selection log, and the quality report into `data/` and `outputs/`.

> Run from the `Data Prep - Derrick` folder so the relative `data/` and `outputs/` paths resolve.

### Where this fits — CRISP-DM Phase 3: Data Preparation

This notebook builds the **final working dataset** from the raw 50K-series source.
CRISP-DM (Week 5) lists five preparation tasks; here is how each maps to the code
below:

| CRISP-DM task (Phase 3) | What we do here |
| --- | --- |
| **Select data** | 2 cities (most stores) x 5 stores (most products) x 10 products x 90 days = **100 complete series / 9,000 rows**; keep full histories, not isolated days |
| **Clean data** | verify 0 duplicates, 0 missing, every series 90/90 days; keep zero-sales and stockout rows (they are real signal, not errors) |
| **Construct data** | derive `availability` and `est_demand` (stockout-adjusted demand) — feature engineering |
| **Integrate / format** | sort by store/product/date, convert `dt` to datetime, write parquet (full) + daily CSV, a data dictionary, and a selection log |

**Missing-data assessment (Week 6).** The subset has **zero missing values**, so the
MCAR / MAR / MNAR imputation machinery is not needed for the recorded fields. But
there is a subtler kind of missingness that *does* matter: **observed sales during a
stockout are censored** — the demand that would have occurred is simply not recorded,
and whether it is "missing" depends on its own size (high-demand items are the ones
that sell out). That is the **MNAR** (Missing Not At Random) pattern from Week 6,
which is *non-ignorable*: you cannot repair it by dropping rows or filling means. Our
`est_demand` column is a transparent, assumption-based reconstruction of it, and —
following the proposal — we treat it as a rough proxy while keeping observed
`sale_amount` as the primary series.

**Selection is a sampling choice, not cleaning.** We deliberately chose the densest
cities/stores and the lowest-id products, so the 100 series are an *illustrative
convenience sample*, not a random draw — a documented limit on how far the results
generalize.

In [1]:
# --- run from anywhere: switch to the data-prep folder so relative paths resolve ---
# (notebooks live here, but the shared data/subset and Data/ folders are under
#  'Data Prep - Derrick'; this makes data/subset/..., ../Data/..., outputs/... all work.)
import os
from pathlib import Path
_cwd = Path.cwd().resolve()
_root = next((p for p in (_cwd, *_cwd.parents) if (p / 'Data Prep - Derrick').is_dir()), None)
if _root is not None:
    os.chdir(_root / 'Data Prep - Derrick')
print('Project data folder located — relative paths ready.')

Project data folder located — relative paths ready.


In [2]:
# MIS 480 group project - build our working subset and run quality checks
# Derrick, data prep
import numpy as np
import pandas as pd
from datasets import load_dataset

In [3]:
# the slice we agreed on in the meeting: 2 cities, 5 stores each, 10 products each
N_CITIES, N_STORES, N_PRODUCTS, N_DAYS = 2, 5, 10, 90

In [4]:
ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K", split="train")
idx = ds.select_columns(["city_id", "store_id", "product_id", "dt"]).to_pandas()

In [5]:
# take the two cities with the most stores
city_stores = idx.groupby("city_id")["store_id"].nunique().sort_values(ascending=False)
cities = sorted(city_stores.head(N_CITIES).index.tolist())

In [6]:
# in each city, the 5 stores with the most products, then 10 products from each
pairs, picks = [], []
for c in cities:
    city_rows = idx[idx.city_id == c]
    stores = city_rows.groupby("store_id")["product_id"].nunique().sort_values(ascending=False).head(N_STORES).index
    for s in sorted(stores):
        prods = sorted(city_rows[city_rows.store_id == s]["product_id"].unique())[:N_PRODUCTS]
        pairs += [(s, p) for p in prods]
        picks.append((c, s, prods))

In [7]:
pairs_set = set(pairs)
print(f"cities: {cities}")
print(f"series picked: {len(pairs_set)}")

cities: [0, 12]
series picked: 100


In [8]:
# grab the full rows (with the hourly arrays) for just those store-product pairs
keep = idx.set_index(["store_id", "product_id"]).index.isin(pairs_set)
sub = ds.select(np.where(keep)[0].tolist()).to_pandas()
sub["dt"] = pd.to_datetime(sub["dt"])
sub = sub.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)

In [9]:
# quality checks
report = []
def note(x):
    print(x); report.append(str(x))

In [10]:
note(f"rows: {len(sub)}")
note(f"series: {sub.groupby(['store_id','product_id']).ngroups}")
note(f"duplicates: {sub.duplicated(['store_id','product_id','dt']).sum()}")
dps = sub.groupby(['store_id','product_id'])['dt'].nunique()
note(f"days per series min/max: {dps.min()}/{dps.max()}")
note(f"dates: {sub['dt'].min().date()} to {sub['dt'].max().date()}")
note("missing values:")
for col in sub.columns:
    if col not in ("hours_sale", "hours_stock_status"):
        note(f"  {col}: {sub[col].isna().sum()}")
note(f"sale_amount min/mean/max: {sub.sale_amount.min():.2f}/{sub.sale_amount.mean():.2f}/{sub.sale_amount.max():.2f}")
note(f"discount min/max: {sub.discount.min():.2f}/{sub.discount.max():.2f}")
note(f"rows with a discount: {(sub.discount < 1.0).sum()}")
note(f"holiday rows: {sub.holiday_flag.sum()}  promo rows: {sub.activity_flag.sum()}")
note(f"rows with a stockout: {(sub.stock_hour6_22_cnt > 0).sum()} ({(sub.stock_hour6_22_cnt>0).mean()*100:.1f}%)")

rows: 9000
series: 100
duplicates: 0
days per series min/max: 90/90
dates: 2024-03-28 to 2024-06-25
missing values:
  city_id: 0
  store_id: 0
  management_group_id: 0
  first_category_id: 0
  second_category_id: 0
  third_category_id: 0
  product_id: 0
  dt: 0
  sale_amount: 0
  stock_hour6_22_cnt: 0
  discount: 0
  holiday_flag: 0
  activity_flag: 0
  precpt: 0
  avg_temperature: 0
  avg_humidity: 0
  avg_wind_level: 0
sale_amount min/mean/max: 0.00/1.22/25.80
discount min/max: 0.44/1.00
rows with a discount: 4863
holiday rows: 3100  promo rows: 3577
rows with a stockout: 3953 (43.9%)


## Stockout-adjusted demand estimate

`sale_amount` only counts sales that actually happened. When an item was out of stock no sale could be recorded, so it understates real demand. We add an `est_demand` column here — using the same simple correction as the EDA notebook — so the whole team shares one demand definition instead of each person re-deriving it. During business hours (6:00–22:00 = 16 one-hour slots) we scale observed sales up by how much of the day the item was actually in stock.

In [11]:
# stockout-adjusted demand estimate (added to both saved files below)
# business-hour window used by stock_hour6_22_cnt: 6:00-22:00 = 16 one-hour slots
BUSINESS_HOURS = 16

# share of the business day the item was in stock
sub["availability"] = (BUSINESS_HOURS - sub["stock_hour6_22_cnt"]) / BUSINESS_HOURS

# scale observed sales up by availability to estimate the demand we would have seen
# if the item had been in stock all day. days out of stock the whole business day
# give us nothing to scale from, so we leave their estimate blank (NaN).
sub["est_demand"] = np.nan
in_stock = sub["availability"] > 0
sub.loc[in_stock, "est_demand"] = sub.loc[in_stock, "sale_amount"] / sub.loc[in_stock, "availability"]

note(f"est_demand mean {sub['est_demand'].mean():.2f} vs observed sale_amount mean {sub['sale_amount'].mean():.2f}")
note(f"days out of stock all business hours (est_demand blank): {int((sub['availability']==0).sum())}")

est_demand mean 1.81 vs observed sale_amount mean 1.22
days out of stock all business hours (est_demand blank): 370


In [12]:
# save the subset and the documentation
sub.to_parquet("data/subset/freshretail_subset.parquet", index=False)
sub.drop(columns=["hours_sale", "hours_stock_status"]).to_csv("data/subset/freshretail_subset_daily.csv", index=False)

In [13]:
data_dict = [
    ("city_id", "city, encoded"),
    ("store_id", "store, encoded"),
    ("management_group_id", "store management group, encoded"),
    ("first_category_id", "top product category"),
    ("second_category_id", "mid product category"),
    ("third_category_id", "detailed product category"),
    ("product_id", "product, encoded"),
    ("dt", "date of the daily record"),
    ("sale_amount", "daily sales, normalized (not units or revenue) - target"),
    ("hours_sale", "24 hourly normalized sales values"),
    ("stock_hour6_22_cnt", "out-of-stock hours during business hours (6-22)"),
    ("hours_stock_status", "24 hourly flags, 1 = out of stock"),
    ("discount", "price ratio, 1.0 = no discount"),
    ("holiday_flag", "1 if holiday"),
    ("activity_flag", "1 if a promo is running"),
    ("precpt", "precipitation"),
    ("avg_temperature", "average temperature"),
    ("avg_humidity", "average humidity"),
    ("avg_wind_level", "average wind"),
    ("availability", "share of business hours (6-22) in stock = (16 - stock_hour6_22_cnt)/16"),
    ("est_demand", "stockout-adjusted demand = sale_amount / availability; blank if out of stock all day"),
]
pd.DataFrame(data_dict, columns=["variable", "description"]).to_csv("outputs/data_dictionary.csv", index=False)

In [14]:
with open("outputs/subset_selection.txt", "w") as f:
    f.write(f"cities (most stores): {cities}\n")
    f.write(f"{N_STORES} stores per city (most products), {N_PRODUCTS} products per store\n\n")
    for c, s, prods in picks:
        f.write(f"city {c} | store {s} | products: {prods}\n")

In [15]:
with open("outputs/quality_report.txt", "w") as f:
    f.write("\n".join(report))

In [16]:
print("saved subset and output files")

saved subset and output files


---
**Pipeline:** 01 inspect → **02 subset** → 03 EDA → 04 modeling → 05 classification.

**Phase 3 outcome.** A clean 100-series / 9,000-row subset with a data dictionary and the engineered `availability` / `est_demand` features, saved to `data/subset/`. This feeds the **EDA (notebook 03)** and the **modeling (notebook 04)**.